In [1]:
from tqdm.auto import tqdm

/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_en/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import openai
import os
openai.api_key = os.environ["OPENAI_API_KEY"]

In [3]:
def prompt(node_text: str, question: str):
    return f"""Can the question "{question}" be answered using only the following facts: "{node_text}"? Answer with yes or no."""

def api_prompt(prompt: str):
    return [
        {"role": "system", "content": "You are a truthful assistant, judging if a question can be answered by some given facts. To be marked as answereable, the question should be answerable by the given facts only and not require any additional resources. You only reply with yes or no."},
        {"role": "user", "content": prompt},
    ]

def api_completion(node_text: str, question: str):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt(prompt(node_text, question))
    )

In [4]:
import json

with open("../../../resources/en/reimburse/generated/train_questions_v2.json", "r") as f:
    data = json.load(f)

print(len(data), "samples")

800 samples


In [5]:
import time

In [10]:
keys = set()
for sample in filtered_questions:
    keys.add(sample['key'])
print(len(keys))

774


In [12]:
for sample in filtered_questions:
    assert "judgement" in sample

In [11]:
# filtered_questions = []
# import traceback

for sample_idx, sample_key in tqdm(enumerate(data)):
    done = False
    while not done:
        if sample_key in keys:
            done = True
        else:
            try:
                sample = data[sample_key]
                dialog_node_key = sample['dialog_node_key']
                question = sample['text']
                node_text = sample['node_text']
                completion = api_completion(node_text, question)
                judgement = completion.get("choices")[0].get("message")["content"]
                sample['judgement'] = judgement
                filtered_questions.append(sample)
                keys.add(sample_key)
                done = True
                # assert len(filtered_questions) == sample_idx + 1
            except:
                # traceback.print_exc()
                print("Waiting")
                time.sleep(30)

800it [00:33, 23.74it/s] 


In [8]:
filtered_questions

[{'dialog_node_key': 16351700401033947,
  'key': '1691659248282345',
  'text': 'What are some reasons for reimbursable flights?',
  'node_text': 'Flights are reimbursable there is a compelling  business or economic reason, e.g.: Ability to attend multiple sequential meetings Health reasons Saving work time Flying is less expensive ',
  'node_type': 'infoNode',
  'judgement': 'Yes'},
 {'dialog_node_key': 16351700401033947,
  'key': '16916592482823682',
  'text': 'Can attending multiple sequential meetings be a reason for reimbursable flights?',
  'node_text': 'Flights are reimbursable there is a compelling  business or economic reason, e.g.: Ability to attend multiple sequential meetings Health reasons Saving work time Flying is less expensive ',
  'node_type': 'infoNode',
  'judgement': 'Yes'},
 {'dialog_node_key': 16351700401033947,
  'key': '16916592482823718',
  'text': 'Are health reasons considered for reimbursable flights?',
  'node_text': 'Flights are reimbursable there is a com

In [13]:
with open("../../../resources/en/reimburse/generated/train_questions_v2_filtered_chatgpt.json", "w") as f:
    json.dump(filtered_questions, f)

In [15]:
completion.get("choices")[0].get("message")[]

'Yes'

In [15]:
import json

with open("../../../resources/en/reimburse/generated/train_questions_v2_ling.json", "r") as f:
    data = json.load(f)

print(len(data), "samples")

filtered_questions = []
keys = set()

for sample_idx, sample_key in tqdm(enumerate(data)):
    done = False
    while not done:
        if sample_key in keys:
            done = True
        else:
            try:
                sample = data[sample_key]
                dialog_node_key = sample['dialog_node_key']
                question = sample['text']
                node_text = sample['node_text']
                completion = api_completion(node_text, question)
                judgement = completion.get("choices")[0].get("message")["content"]
                sample['judgement'] = judgement
                filtered_questions.append(sample)
                keys.add(sample_key)
                done = True
                # assert len(filtered_questions) == sample_idx + 1
            except:
                # traceback.print_exc()
                print("Waiting")
                time.sleep(30)

with open("../../../resources/en/reimburse/generated/train_questions_v2_ling_filtered_chatgpt.json", "w") as f:
    json.dump(filtered_questions, f)


3062 samples


7it [00:11,  1.52s/it]

Waiting


In [3]:
import json

filtered_questions = {}
# respones = set()
with open("../../../resources/en/reimburse/generated/train_questions_v2_filtered_chatgpt.json", "r") as f:
    data = json.load(f)
    print(len(data))
    for sample in data:
        judgement = sample['judgement']
        if judgement == "Yes." or judgement == "Yes":
            filtered_questions[sample["key"]] = sample
print(len(filtered_questions))

549
549


In [4]:
with open("../../../resources/en/reimburse/generated/train_questions_v2_filtered_chatgpt.json", "w") as f:
    json.dump(filtered_questions, f)

In [3]:
!ls ../../../resources/en/reimburse/generated/

chatgpt
train_answers.json
train_questions_v1.json
train_questions_v1_ling.json
train_questions_v1_ling_with_judgement.json
train_questions_v1_with_judgement.json
train_questions_v2_filtered_chatgpt.json
train_questions_v2.json
train_questions_v2_ling.json
train_questions_v2_ling_with_judgement.json
train_questions_v2_with_judgement.json
train_questions_v3.json
train_questions_v3_ling.json
train_questions_v3_with_judgement.json


In [3]:
import json

filtered_questions = {}
# respones = set()
with open("../../../resources/en/reimburse/generated/train_questions_v2_ling_filtered_chatgpt.json", "r") as f:
    data = json.load(f)
    print(len(data))
    for sample in data:
        judgement = sample['judgement']
        # respones.add(judgement)
        if judgement.startswith("Yes"):
            filtered_questions[sample["key"]] = sample
print(len(filtered_questions))

3062
663


In [4]:
with open("../../../resources/en/reimburse/generated/train_questions_v2_ling_filtered_chatgpt.json", "w") as f:
    json.dump(filtered_questions, f)